In [8]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score,f1_score,precision_score,accuracy_score
from sklearn.feature_selection import SelectKBest,f_classif

In [2]:
df=pd.read_csv("../data/data.csv")

In [3]:
df=df.drop(columns="id")
df=df.drop(columns="Unnamed: 32")

In [4]:
df['diagnosis']=df['diagnosis'].map({"B":0,"M":1})

In [5]:
X=df.drop(columns='diagnosis')
Y=df['diagnosis']

In [22]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=2,stratify=Y)

In [9]:
pipe=Pipeline([
    ("scaler",StandardScaler()),
    ("selector", SelectKBest(score_func=f_classif, k=10)),
    ("model", LogisticRegression(C=1, max_iter=100))
])

In [19]:
param = {
    "selector__k": [5, 10, 15, 20, 25, "all"]
}
from sklearn.model_selection import GridSearchCV
grid=GridSearchCV(pipe,param,cv=5,n_jobs=-1,scoring="recall")
grid.fit(X_train,Y_train)
print(grid.best_score_)
print(grid.best_params_)

0.9647058823529411
{'selector__k': 20}


In [20]:
best_pipe=grid.best_estimator_
print(grid.best_score_)
print(grid.best_params_)

0.9647058823529411
{'selector__k': 20}


In [21]:
prediction = best_pipe.predict(X_test)

print("Accuracy :", accuracy_score(Y_test, prediction))
print("Precision:", precision_score(Y_test, prediction))
print("Recall   :", recall_score(Y_test, prediction))
print("F1       :", f1_score(Y_test, prediction))

Accuracy : 0.956140350877193
Precision: 0.9302325581395349
Recall   : 0.9523809523809523
F1       : 0.9411764705882353


Compared with our original model with 30 features, Recall is same, but the other have dropped one or 2 units. So no need to go ahead with feature selection.

In [24]:
selector = best_pipe.named_steps["selector"]

selected_features = X.columns[selector.get_support()]

print(selected_features.tolist())

['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'radius_se', 'perimeter_se', 'area_se', 'concave points_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst']


Now we try feature extraction, though we lose our original columns and interpretation of prediction becomes difficult as to which feature affected it most.

In [25]:
from sklearn.decomposition import PCA
pca_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("model", LogisticRegression(C=1, max_iter=100))
])

In [26]:
param = {
    "pca__n_components": [5, 10, 15, 20, 25, 30]
}
grid_pca = GridSearchCV(
    pca_pipe,
    param,
    scoring="recall",
    cv=5,
    n_jobs=-1
)

grid_pca.fit(X_train, Y_train)

print("Best parameters:", grid_pca.best_params_)
print("Best CV recall:", grid_pca.best_score_)

Best parameters: {'pca__n_components': 5}
Best CV recall: 0.9588235294117646


In [27]:
best_pca = grid_pca.best_estimator_

prediction = best_pca.predict(X_test)

print("Accuracy :", accuracy_score(Y_test, prediction))
print("Precision:", precision_score(Y_test, prediction))
print("Recall   :", recall_score(Y_test, prediction))
print("F1       :", f1_score(Y_test, prediction))

Accuracy : 0.956140350877193
Precision: 0.9302325581395349
Recall   : 0.9523809523809523
F1       : 0.9411764705882353


Again no increase in recall, but others decreased. same as feature selection.

So we conclude our simple Logistic regression model is far better.